# BossFx Voice: voice-first financial literacy on N-ATLaS

Telegram bot: a user sends a **voice note in Yoruba, Hausa, Igbo or Nigerian English** → the official **N-ATLaS ASR** model for that language transcribes it → **N-ATLaS LLM** answers as a financial-literacy tutor (money basics, forex education, risk, scam awareness) → reply in the same language. Every interaction is logged anonymously for real-world validation.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or better).
2. On Hugging Face, open each model page and click **Agree / Access repository**: `NCAIR1/N-ATLaS`, `NCAIR1/Yoruba-ASR`, `NCAIR1/Hausa-ASR`, `NCAIR1/Igbo-ASR`, `NCAIR1/NigerianAccentedEnglish`.
3. Colab sidebar 🔑 **Secrets**: add `HF_TOKEN` (Hugging Face read token) and `TELEGRAM_TOKEN` (from @BotFather). Turn on notebook access for both.
4. Run all cells top to bottom.

In [ ]:
# 1. Install
!pip -q install -U transformers accelerate bitsandbytes pyTelegramBotAPI librosa soundfile pandas
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Config, secrets, persistent storage on Google Drive
import os, json, time, hashlib, subprocess, tempfile, threading, datetime, uuid
from google.colab import userdata, drive
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
TELEGRAM_TOKEN = userdata.get("TELEGRAM_TOKEN")
login(HF_TOKEN)

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/BossFxVoice"
os.makedirs(DATA_DIR, exist_ok=True)
LOG_PATH = f"{DATA_DIR}/interactions.jsonl"

USE_4BIT = True   # True fits a free T4. Set False on an A100/L4 to run full fp16.
print("Logging to", LOG_PATH)

In [ ]:
# 3. Load N-ATLaS LLM + the four official N-ATLaS ASR models
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

LLM_ID = "NCAIR1/N-ATLaS"
ASR_IDS = {
    "yo": "NCAIR1/Yoruba-ASR",
    "ha": "NCAIR1/Hausa-ASR",
    "ig": "NCAIR1/Igbo-ASR",
    "en": "NCAIR1/NigerianAccentedEnglish",
}
LANG_NAMES = {"yo": "Yoruba", "ha": "Hausa", "ig": "Igbo", "en": "English"}

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
if USE_4BIT:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, quantization_config=bnb, device_map="auto")
else:
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16, device_map="auto")
llm.eval()

asr = {}
for lang, mid in ASR_IDS.items():
    asr[lang] = pipeline("automatic-speech-recognition", model=mid,
                         torch_dtype=torch.float16, device=0, chunk_length_s=30)
    print("Loaded ASR:", mid)
print("All models ready. GPU memory used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# 4. Core pipeline: speech -> text (N-ATLaS ASR) -> answer (N-ATLaS LLM)
SYSTEM = """You are BossFx Voice, a friendly financial-literacy tutor built by BossFx Academy on N-ATLaS. You help everyday Nigerians understand money: saving, budgeting, inflation, forex trading basics, risk management, and how to spot investment and trading scams.
Rules:
- Always reply in {language}, using simple words a beginner understands.
- Keep answers short: 3 to 6 sentences, or a short numbered list.
- You teach. You never give trading signals, never tell anyone what to buy or sell, and never promise profits.
- If someone mentions guaranteed returns, doubling money, or sending money to an "account manager", clearly warn that it is very likely a scam.
- When trading comes up, remind people to only risk money they can afford to lose.
- If you are not sure about something, say so honestly."""

gpu_lock = threading.Lock()

def ask_natlas(question, lang, history=None):
    messages = [{"role": "system", "content": SYSTEM.format(language=LANG_NAMES[lang])}]
    if history:
        messages += history[-4:]
    messages.append({"role": "user", "content": question})
    text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
        date_string=datetime.datetime.now().strftime("%d %b %Y"))
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(llm.device)
    with gpu_lock, torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=350, do_sample=False,
                           repetition_penalty=1.12, use_cache=True,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def to_wav16k(src):
    dst = src.rsplit(".", 1)[0] + ".wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", "16000", dst], check=True)
    return dst

def transcribe(path, lang):
    import librosa
    audio, _ = librosa.load(to_wav16k(path), sr=16000)
    with gpu_lock:
        result = asr[lang]({"raw": audio, "sampling_rate": 16000})
    return result["text"].strip(), round(len(audio) / 16000, 1)

log_lock = threading.Lock()
def log_event(rec):
    rec["ts"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with log_lock, open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def anon(uid):
    # Telegram IDs are hashed: no names or phone numbers are stored
    return hashlib.sha256(f"bossfx-voice-{uid}".encode()).hexdigest()[:12]

In [ ]:
# 5. Smoke test the LLM in all four languages (run once, check the answers make sense)
tests = {
    "en": "Someone on WhatsApp promised to double my 50,000 naira in one week. Is it safe?",
    "yo": "Kí ni forex trading?",
    "ha": "Ta yaya zan fara ajiyar kudi?",
    "ig": "Kedu ihe bụ forex trading?",
}
for lang, q in tests.items():
    t0 = time.time()
    a = ask_natlas(q, lang)
    print(f"--- {LANG_NAMES[lang]} ({time.time()-t0:.0f}s) ---\nQ: {q}\nA: {a}\n")

In [ ]:
# 6. Telegram bot (runs in the background so you can keep using the notebook)
import telebot
from telebot import types

bot = telebot.TeleBot(TELEGRAM_TOKEN, parse_mode=None)
user_lang, user_hist, pending = {}, {}, {}

WELCOME = ("👋 Welcome to BossFx Voice!\n\n"
           "Ask me anything about money, saving, forex trading basics or scams, "
           "by VOICE NOTE in Yoruba, Hausa, Igbo or English.\n\n"
           "ℹ️ This is education, not financial advice. Questions are logged anonymously "
           "(no names or numbers) to improve the service.\n\n"
           "Powered by N-ATLaS, an initiative of the Federal Ministry of Communications, "
           "Innovation and Digital Economy, powered by Awarri Technologies.\n\n"
           "Choose your language 👇")
READY = {
    "yo": "Ẹ ṣé! Fi ìbéèrè rẹ ránṣẹ́ pẹ̀lú voice note. 🎙️\n(Send your question as a voice note.)",
    "ha": "Na gode! Aiko da tambayarka ta saƙon murya (voice note). 🎙️\n(Send your question as a voice note.)",
    "ig": "Daalụ! Ziga ajụjụ gị site na voice note. 🎙️\n(Send your question as a voice note.)",
    "en": "Great! Send your question as a voice note. 🎙️",
}

def lang_kb():
    kb = types.InlineKeyboardMarkup()
    kb.row(types.InlineKeyboardButton("Yorùbá", callback_data="lang:yo"),
           types.InlineKeyboardButton("Hausa", callback_data="lang:ha"))
    kb.row(types.InlineKeyboardButton("Igbo", callback_data="lang:ig"),
           types.InlineKeyboardButton("English", callback_data="lang:en"))
    return kb

def fb_kb(iid):
    kb = types.InlineKeyboardMarkup()
    kb.row(types.InlineKeyboardButton("👍 Helpful", callback_data=f"fb:up:{iid}"),
           types.InlineKeyboardButton("👎 Not helpful", callback_data=f"fb:down:{iid}"))
    kb.row(types.InlineKeyboardButton("🌍 Change language", callback_data="chg"))
    return kb

@bot.message_handler(commands=["start", "language"])
def on_start(m):
    bot.send_message(m.chat.id, WELCOME, reply_markup=lang_kb())

@bot.callback_query_handler(func=lambda c: c.data.startswith("lang:") or c.data == "chg")
def on_lang(c):
    bot.answer_callback_query(c.id)
    if c.data == "chg":
        return bot.send_message(c.message.chat.id, "Choose your language 👇", reply_markup=lang_kb())
    lang = c.data.split(":")[1]
    user_lang[c.from_user.id] = lang
    user_hist[c.from_user.id] = []
    log_event({"type": "language_set", "user": anon(c.from_user.id), "lang": lang})
    bot.send_message(c.message.chat.id, READY[lang])

@bot.callback_query_handler(func=lambda c: c.data.startswith("fb:"))
def on_feedback(c):
    _, rating, iid = c.data.split(":", 2)
    log_event({"type": "feedback", "interaction_id": iid, "user": anon(c.from_user.id), "rating": rating})
    bot.answer_callback_query(c.id, "Thank you! 🙏")

def answer_and_log(m, lang, input_type, question, t0, asr_s=None, audio_s=None):
    uid = m.from_user.id
    answer = ask_natlas(question, lang, user_hist.get(uid))
    hist = user_hist.setdefault(uid, [])
    hist += [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
    iid = uuid.uuid4().hex[:10]
    body = (f"🗣️ You said: {question}\n\n" if input_type == "voice" else "") + f"💬 {answer}"
    bot.send_message(m.chat.id, body[:4000], reply_markup=fb_kb(iid))
    log_event({"type": "interaction", "interaction_id": iid, "user": anon(uid), "lang": lang,
               "input_type": input_type, "audio_seconds": audio_s, "transcript": question,
               "response": answer, "asr_seconds": asr_s,
               "total_seconds": round(time.time() - t0, 1),
               "asr_model": ASR_IDS[lang] if input_type == "voice" else None, "llm_model": LLM_ID})

def need_lang(m):
    if m.from_user.id not in user_lang:
        bot.send_message(m.chat.id, "Choose your language first 👇", reply_markup=lang_kb())
        return True
    return False

@bot.message_handler(content_types=["voice", "audio"])
def on_voice(m):
    if need_lang(m): return
    lang = user_lang[m.from_user.id]
    t0 = time.time()
    try:
        bot.send_chat_action(m.chat.id, "typing")
        f = m.voice or m.audio
        data = bot.download_file(bot.get_file(f.file_id).file_path)
        with tempfile.NamedTemporaryFile(suffix=".ogg", delete=False) as tmp:
            tmp.write(data); path = tmp.name
        text, secs = transcribe(path, lang)
        asr_s = round(time.time() - t0, 1)
        if not text:
            return bot.send_message(m.chat.id, "I couldn't hear that clearly 🙏 Please try again in a quieter place.")
        bot.send_chat_action(m.chat.id, "typing")
        answer_and_log(m, lang, "voice", text, t0, asr_s, secs)
    except Exception as e:
        log_event({"type": "error", "user": anon(m.from_user.id), "lang": lang, "error": repr(e)})
        bot.send_message(m.chat.id, "Sorry, something went wrong. Please try again. 🙏")

@bot.message_handler(content_types=["text"])
def on_text(m):
    if m.text.startswith("/") or need_lang(m): return
    t0 = time.time()
    try:
        bot.send_chat_action(m.chat.id, "typing")
        answer_and_log(m, user_lang[m.from_user.id], "text", m.text.strip(), t0)
    except Exception as e:
        log_event({"type": "error", "user": anon(m.from_user.id), "error": repr(e)})
        bot.send_message(m.chat.id, "Sorry, something went wrong. Please try again. 🙏")

bot.remove_webhook()
threading.Thread(target=lambda: bot.infinity_polling(timeout=30, long_polling_timeout=30, skip_pending=True),
                 daemon=True).start()
print("✅ BossFx Voice is live:", "https://t.me/" + bot.get_me().username)

In [ ]:
# 7. Live validation report (re-run any time; writes CSV + summary to Google Drive)
import pandas as pd

rows = [json.loads(l) for l in open(LOG_PATH, encoding="utf-8")] if os.path.exists(LOG_PATH) else []
df = pd.DataFrame(rows)
inter = df[df.type == "interaction"].copy() if len(df) else pd.DataFrame()
fb = df[df.type == "feedback"] if len(df) else pd.DataFrame()

if len(inter):
    if len(fb):
        last = fb.drop_duplicates("interaction_id", keep="last").set_index("interaction_id")["rating"]
        inter["rating"] = inter.interaction_id.map(last)
    else:
        inter["rating"] = None
    rated = inter.rating.notna().sum()
    summary = {
        "total_interactions": len(inter),
        "voice_interactions": int((inter.input_type == "voice").sum()),
        "text_interactions": int((inter.input_type == "text").sum()),
        "unique_users": inter.user.nunique(),
        "by_language": inter.lang.map(LANG_NAMES).value_counts().to_dict(),
        "avg_response_seconds": round(inter.total_seconds.mean(), 1),
        "rated": int(rated),
        "helpful_rate_pct": round(100 * (inter.rating == "up").sum() / rated, 1) if rated else None,
        "errors": int((df.type == "error").sum()),
        "first_interaction": inter.ts.min(), "last_interaction": inter.ts.max(),
    }
    inter.to_csv(f"{DATA_DIR}/validation_interactions.csv", index=False)
    json.dump(summary, open(f"{DATA_DIR}/validation_summary.json", "w"), indent=2, default=str)
    print(json.dumps(summary, indent=2, default=str))
    display(inter[["ts", "lang", "input_type", "transcript", "response", "rating", "total_seconds"]].tail(10))
else:
    print("No interactions yet.")